# Spark Optimization for Data Engineering


## Why Optimization Matters

Spark can process huge data…

But:
- Bad code → slow jobs
- Poor design → high cost

👉 Strong pipelines are **correct** first — then **fast**.


## Common issues

- Too many **shuffles**
- **Skew** (one hot key)
- **Small files** on write
- **Recompute** (same lineage executed twice)

👉 Profile before tuning — don’t guess.


## Prerequisites

- PySpark + JDK ([`README.md`](README.md)).


## Spark session


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("SparkOptimization")
    .master("local[*]")
    .getOrCreate()
)


## Repartition vs coalesce

- **`repartition(n)`** — **shuffle** to spread/redistribute partitions (can increase or decrease count).
- **`coalesce(n)`** — **narrow** merge down (no full shuffle; **cannot** safely increase partitions).

👉 Pick based on whether you need a shuffle or cheaper narrow merge.


In [ ]:
df = spark.range(1000)

print("default partitions:", df.rdd.getNumPartitions())

r10 = df.repartition(10)
print("after repartition(10):", r10.rdd.getNumPartitions())

c2 = r10.coalesce(2)
print("after coalesce(2):", c2.rdd.getNumPartitions())


## Caching (IMPORTANT)

**`cache()` / `persist()`** materialize data after first action so **reuse** avoids recomputing lineage.

**Use when:** the same DataFrame is hit multiple times (iterative algorithms, repeated joins).

**Avoid:** caching huge frames you only scan once — memory pressure.


In [ ]:
wide = spark.range(10000)
wide.cache()

# First action materializes cache
print("rows:", wide.count())

# Second action reuses cache (same Spark session)
print("rows again:", wide.count())


## Narrow vs wide transformations

**Narrow** (no shuffle): `filter`, `select`, `withColumn` within partition.

**Wide** (shuffle): `groupBy`, `join`, `repartition`, most aggregations across keys.

👉 Wide stages exchange data across the cluster — usually where time goes.


## Broadcast join


In [ ]:
from pyspark.sql.functions import broadcast

big = spark.range(100).toDF("id")

small_df = spark.createDataFrame([(1, "A"), (2, "B")], ["id", "value"])

big.join(broadcast(small_df), "id").orderBy("id").show()


## Partitioning strategy

Organize stored data by **predicate columns** (often **date**, **region**, **user_id**):

- Fewer files scanned per query
- Better **partition pruning** when filters match folder layout

👉 Align layout with **how** queries filter.


## File formats

| Format | Notes |
|--------|--------|
| CSV | Row-oriented, verbose, slow scans |
| JSON | Semi-structured; flexible but heavier |
| **Parquet** | Columnar, compressed, **predicate pushdown** — default choice for analytics |

👉 Prefer **Parquet** (or Iceberg/Delta on top) for lake/warehouse facts.


## Explain plan

`df.explain()` / `df.explain(mode="cost")` — inspect **logical → physical** plan (shuffle boundaries, joins).

👉 First step when a stage is unexpectedly slow.


In [ ]:
sample = spark.range(50).selectExpr("id", "id * 2 as double_id")

sample.explain(True)


## Practice

1. **Cache** a DataFrame and run **two** actions (`count`, `collect` sample).
2. Compare **`repartition(n)`** vs **`coalesce(n)`** partition counts on the same frame.
3. Run a **broadcast join** where the small side is obvious.


## Assignment

Optimize a pipeline you already built (this chapter or prior notebooks):

- Add **`cache`** where reuse is real
- Reduce unnecessary **shuffles** (fewer wide ops, better keys)
- Reason about **output partitioning** / file sizes on write

**Bonus:** write **Parquet** with sensible partition columns.


## Interview Questions

1. What causes shuffle in Spark?
2. What is broadcast join?
3. Difference between repartition and coalesce?
4. How do you optimize Spark jobs?


## What you just learned

- **Tuning** knobs: partitions, cache, broadcast, storage layout
- **Reading plans** with `explain`

👉 Typical senior IC / staff interview depth.


---

**Next:** **Spark project** — end-to-end pipeline + tuning story for your portfolio.

**Reality check:** you can now pair **correctness** with **performance** vocabulary.


## Optional: stop Spark


In [ ]:
spark.stop()
print("Spark stopped.")


## Your tasks

- [ ] Run **Repartition** → **Explain plan** once with a fresh kernel.
- [ ] **Practice:** cache reuse; repartition vs coalesce counts; broadcast join.
- [ ] **Assignment:** apply **cache + shuffle reduction + partitioning** to one pipeline; **bonus:** Parquet.
- [ ] Answer interview questions; sketch **one** skew mitigation (salting, adaptive execution).
